# trend_participation_foldlocal_target_v1

Purpose: convert the failed 2024 high-trend / low-volatility participation diagnosis into fold-local training/checkpoint target candidates. The previous runtime overlay screen showed post-hoc trend floors make `wf_2023_2024` worse, so this notebook does not promote an execution overlay. It tests whether the same state can be represented as non-leaking target labels that pass scope, lift, and BC-readiness gates before any BC/PPO run.

Governance:

- **Baselines**: canonical `v19reg_v22_budget_usage_bull_guard_v1`; direct comparison `trend_participation_guard_screen_v1` and `cap10_sequence_risk_target_cap_bc_screen_v1/baseline_no_sequence_cap_target_contract.csv`.
- **Role**: `diagnose`.
- **Target window**: fold-local train rows for `wf_2022_train` and `wf_2023_2024_train`; eval windows remain excluded.
- **Changed layer**: `model_selection` / `training_target`.
- **External reference**: Qlib workflow and Qlib paper for repeatable ML evaluation, Probability of Backtest Overfitting for promotion discipline, time-series momentum and momentum-crash papers for trend state risk, volatility-managed portfolios for volatility guard design, and FinRL for environment/reward/action separation. The momentum-crash reference is specifically Daniel and Moskowitz JFE 2016 / DOI `10.1016/j.jfineco.2015.12.002`.
- **Hypothesis**: train-quantile high-trend / low-volatility rows can be encoded as sparse target-position lifts without leaking into evaluation rows; if target labels are too broad, degenerate, or erased by downtrend priority, do not train.
- **Acceptance criteria**: target scope audit has zero eval overlap, target lift audit has real current positive lift, BC readiness passes label-distribution gates, and no runtime promotion is allowed from this card.
- **Failure mode**: no target variant passes scope/lift/readiness, or trend rows are too broad/sparse, meaning the state definition must be revised before training.
- **Comparison-table row key**: `trend_participation_foldlocal_target_v1`.

Historical reflection:

- Closest success: `cap10_foldlocal_joint_target_contract_v1` showed fold-local joint targets can pass leakage/lift/readiness gates and remain a training recipe, not a runtime promotion.
- Closest failure: `trend_participation_guard_screen_v1` showed runtime trend floors worsen `wf_2023_2024` relative return and drawdown.
- Why this is not a duplicate: this notebook changes the artifact from execution overlay to target contract, derives thresholds from fold train features, and audits leakage/lift/label distribution before training.


In [1]:
from __future__ import annotations

import json
import math
import os
import sys
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd()
while not (ROOT / 'AGENTS.md').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

from src.soxl_quant.app.state_objective_training_targets import (
    TrendParticipationTargetConfig,
    apply_trend_participation_target_gate,
    audit_state_objective_target_lift,
    audit_state_objective_target_scope,
    score_state_objective_bc_readiness,
    summarize_state_objective_bc_readiness,
    summarize_state_objective_target_lift_audit,
    summarize_state_objective_target_scope_audit,
)
from src.soxl_quant.config import load_profile
from src.soxl_quant.data import build_dataset, init_qlib, segment_frame
from src.soxl_quant.settings import QLIB_DATA_PATH

EXPERIMENT = 'trend_participation_foldlocal_target_v1'
CARD = ROOT / 'runtime' / 'model_selection' / 'card' / EXPERIMENT
CARD.mkdir(parents=True, exist_ok=True)
SOURCE_TARGETS = ROOT / 'runtime/model_selection/card/cap10_sequence_risk_target_cap_bc_screen_v1/baseline_no_sequence_cap_target_contract.csv'
SOURCE_TREND_SCREEN = ROOT / 'runtime/model_selection/card/trend_participation_guard_screen_v1/selection.json'
PROFILE_NAME = 'sparse_release_bc_ppo_screen_v1'
TARGET_SCOPE = 'fold_local'

EXTERNAL_SOURCES = [
    'https://github.com/microsoft/qlib',
    'https://arxiv.org/abs/2009.11189',
    'https://doi.org/10.21314/JCF.2016.322',
    'https://doi.org/10.1016/j.jfineco.2011.11.003',
    'https://doi.org/10.1016/j.jfineco.2015.12.002',
    'https://doi.org/10.1111/jofi.12513',
    'https://arxiv.org/abs/2011.09607',
    'https://github.com/AI4Finance-Foundation/FinRL',
]

RECIPES = [
    {
        'candidate': 'base_foldlocal_contract',
        'enabled': False,
        'note': 'Unchanged fold-local target contract.',
    },
    {
        'candidate': 'trend_q70_target45_guard',
        'enabled': True,
        'min_target_position': 0.45,
        'max_target_position': 0.55,
        'max_teacher_position': 0.35,
        'anchor_q': 0.70,
        'mom_q': 0.70,
        'trend_slope_q': 0.65,
        'trend_gap_q': 0.65,
        'vix_q': 0.50,
        'realized_vol_q': 0.75,
        'drawdown_q': 0.40,
        'objective_weight_bonus': 4.0,
        'exclude_downtrend': True,
        'note': 'Train-quantile high-trend low-vol target lift, conservative floor.',
    },
    {
        'candidate': 'trend_q75_target50_guard',
        'enabled': True,
        'min_target_position': 0.50,
        'max_target_position': 0.60,
        'max_teacher_position': 0.35,
        'anchor_q': 0.75,
        'mom_q': 0.75,
        'trend_slope_q': 0.70,
        'trend_gap_q': 0.70,
        'vix_q': 0.50,
        'realized_vol_q': 0.72,
        'drawdown_q': 0.45,
        'objective_weight_bonus': 5.0,
        'exclude_downtrend': True,
        'note': 'Mid-strength target lift with tighter realized-vol guard.',
    },
    {
        'candidate': 'trend_q80_target55_guard',
        'enabled': True,
        'min_target_position': 0.55,
        'max_target_position': 0.65,
        'max_teacher_position': 0.30,
        'anchor_q': 0.80,
        'mom_q': 0.80,
        'trend_slope_q': 0.75,
        'trend_gap_q': 0.75,
        'vix_q': 0.48,
        'realized_vol_q': 0.70,
        'drawdown_q': 0.45,
        'objective_weight_bonus': 6.0,
        'exclude_downtrend': True,
        'note': 'Sparse high-conviction target lift.',
    },
]


def json_safe(value: Any) -> Any:
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(v) for v in value]
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.bool_,)):
        return bool(value)
    if isinstance(value, (np.floating,)):
        value = float(value)
    if isinstance(value, float):
        return value if math.isfinite(value) else None
    if isinstance(value, pd.Timestamp):
        return str(value)
    if isinstance(value, Path):
        return str(value)
    return value


def write_json(path: Path, payload: dict[str, Any]) -> None:
    path.write_text(json.dumps(json_safe(payload), indent=2, sort_keys=True, allow_nan=False) + chr(10), encoding='utf-8')

print('root', ROOT)
print('card', CARD)


root /Users/rziiiii/.openclaw/workspace
card /Users/rziiiii/.openclaw/workspace/runtime/model_selection/card/trend_participation_foldlocal_target_v1


In [2]:
def derive_folds(targets: pd.DataFrame) -> list[dict[str, str]]:
    required = ['fold', 'train_start', 'train_end', 'eval_start', 'eval_end']
    missing = sorted(set(required) - set(targets.columns))
    if missing:
        raise ValueError(f'source targets missing fold metadata: {missing}')
    folds = []
    for fold, group in targets.groupby('fold', sort=True):
        unique = group[required].drop_duplicates()
        if len(unique) != 1:
            raise ValueError(f'fold {fold} has inconsistent boundary rows')
        row = unique.iloc[0]
        folds.append({
            'name': str(fold),
            'train_start': str(row['train_start']),
            'train_end': str(row['train_end']),
            'eval_start': str(row['eval_start']),
            'eval_end': str(row['eval_end']),
        })
    return folds


def effective_target_position(targets: pd.DataFrame) -> pd.Series:
    frame = targets.copy()
    for col in ['teacher_position', 'rebound_target_position', 'risk_target_position']:
        frame[col] = pd.to_numeric(frame[col], errors='coerce').fillna(0.0)
    for col in ['is_rebound_cluster', 'is_downtrend_state']:
        frame[col] = frame[col].fillna(False).astype(bool)
    return pd.Series(
        np.where(
            frame['is_downtrend_state'],
            frame['risk_target_position'],
            np.where(frame['is_rebound_cluster'], frame['rebound_target_position'], frame['teacher_position']),
        ),
        index=frame.index,
    ).clip(0.0, 1.0)


def target_action_distribution(targets: pd.DataFrame) -> pd.DataFrame:
    if targets.empty:
        return pd.DataFrame(columns=['profile', 'samples', 'unique_actions', 'dominant_action_share', 'action_entropy'])
    frame = targets.copy()
    frame['target_action'] = np.round(effective_target_position(frame) / 0.05).astype(int).clip(0, 20)
    rows = []
    for profile, group in frame.groupby('fold', sort=True):
        counts = group['target_action'].value_counts(normalize=True)
        rows.append({
            'profile': str(profile),
            'samples': int(len(group)),
            'unique_actions': int(group['target_action'].nunique()),
            'dominant_action': int(counts.index[0]) if not counts.empty else 0,
            'dominant_action_share': float(counts.iloc[0]) if not counts.empty else 1.0,
            'action_entropy': float(-(counts * np.log(counts + 1e-12)).sum()) if not counts.empty else 0.0,
            'rebound_rows': int(group['is_rebound_cluster'].fillna(False).astype(bool).sum()),
            'downtrend_rows': int(group['is_downtrend_state'].fillna(False).astype(bool).sum()),
            'trend_target_rows': int(group.get('trend_participation_target_state', pd.Series(False, index=group.index)).fillna(False).astype(bool).sum()),
        })
    counts = frame['target_action'].value_counts(normalize=True)
    rows.append({
        'profile': 'all_folds_trend_contract',
        'samples': int(len(frame)),
        'unique_actions': int(frame['target_action'].nunique()),
        'dominant_action': int(counts.index[0]) if not counts.empty else 0,
        'dominant_action_share': float(counts.iloc[0]) if not counts.empty else 1.0,
        'action_entropy': float(-(counts * np.log(counts + 1e-12)).sum()) if not counts.empty else 0.0,
        'rebound_rows': int(frame['is_rebound_cluster'].fillna(False).astype(bool).sum()),
        'downtrend_rows': int(frame['is_downtrend_state'].fillna(False).astype(bool).sum()),
        'trend_target_rows': int(frame.get('trend_participation_target_state', pd.Series(False, index=frame.index)).fillna(False).astype(bool).sum()),
    })
    return pd.DataFrame(rows)


def qvalue(frame: pd.DataFrame, column: str, q: float) -> float:
    return float(pd.to_numeric(frame[column], errors='coerce').quantile(float(q)))


def config_from_recipe(recipe: dict[str, Any], train_features: pd.DataFrame) -> TrendParticipationTargetConfig:
    if not recipe.get('enabled', False):
        return TrendParticipationTargetConfig(enabled=False)
    return TrendParticipationTargetConfig(
        enabled=True,
        min_target_position=float(recipe['min_target_position']),
        max_target_position=float(recipe['max_target_position']),
        max_teacher_position=float(recipe['max_teacher_position']),
        min_anchor_gap_180=qvalue(train_features, 'anchor_gap_180', recipe['anchor_q']),
        min_mom_63=qvalue(train_features, 'mom_63', recipe['mom_q']),
        min_trend_slope_63=qvalue(train_features, 'trend_slope_63', recipe['trend_slope_q']),
        min_trend_gap_63=qvalue(train_features, 'trend_gap_63', recipe['trend_gap_q']),
        max_vix_percentile_252=qvalue(train_features, 'vix_percentile_252', recipe['vix_q']),
        max_realized_vol_63=qvalue(train_features, 'realized_vol_63', recipe['realized_vol_q']),
        min_drawdown_63=qvalue(train_features, 'drawdown_63', recipe['drawdown_q']),
        objective_weight_bonus=float(recipe['objective_weight_bonus']),
        exclude_downtrend=bool(recipe['exclude_downtrend']),
    )


In [3]:
base_targets = pd.read_csv(SOURCE_TARGETS)
base_targets['datetime'] = pd.to_datetime(base_targets['datetime'])
folds = derive_folds(base_targets)
trend_screen = json.loads(SOURCE_TREND_SCREEN.read_text(encoding='utf-8'))

profile = load_profile(PROFILE_NAME)
init_qlib(QLIB_DATA_PATH)
feature_frames = {}
for fold in folds:
    dataset = build_dataset(
        train_seg=(fold['train_start'], fold['train_end']),
        valid_seg=(fold['train_start'], fold['train_end']),
        test_seg=(fold['train_start'], fold['train_end']),
        feature_packs=tuple(profile.feature.packs),
        profile_name=profile.name,
    )
    features = segment_frame(dataset, 'train').reset_index().rename(columns={'index': 'datetime'})
    features['datetime'] = pd.to_datetime(features['datetime'])
    feature_frames[fold['name']] = features

pd.DataFrame([{
    'fold': fold['name'],
    'target_rows': int((base_targets['fold'].astype(str) == fold['name']).sum()),
    'feature_rows': len(feature_frames[fold['name']]),
    'train_start': fold['train_start'],
    'train_end': fold['train_end'],
    'eval_start': fold['eval_start'],
    'eval_end': fold['eval_end'],
} for fold in folds])


[88643:MainThread](2026-05-26 13:44:47,227) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[88643:MainThread](2026-05-26 13:44:48,050) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[88643:MainThread](2026-05-26 13:44:48,051) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/.openclaw/workspace/data/qlib/soxl')}


[88643:MainThread](2026-05-26 13:44:48,216) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | Loading data Done


[88643:MainThread](2026-05-26 13:44:48,218) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | fit & process data Done


[88643:MainThread](2026-05-26 13:44:48,221) INFO - qlib.timer - [log.py:127] - Time cost: 0.005s | Init data Done


[88643:MainThread](2026-05-26 13:44:48,617) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | Loading data Done


[88643:MainThread](2026-05-26 13:44:48,620) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | fit & process data Done


[88643:MainThread](2026-05-26 13:44:48,622) INFO - qlib.timer - [log.py:127] - Time cost: 0.005s | Init data Done


,fold,target_rows,feature_rows,train_start,train_end,eval_start,eval_end
0,wf_2022_train,505,254,2020-01-02,2021-12-31,2022-01-03,2022-12-30
1,wf_2023_2024_train,756,505,2020-01-02,2022-12-30,2023-01-03,2024-12-31


In [4]:
candidate_targets = {}
config_rows = []
variant_rows = []
for recipe in RECIPES:
    candidate = recipe['candidate']
    fold_frames = []
    for fold in folds:
        fold_name = fold['name']
        fold_targets = base_targets[base_targets['fold'].astype(str).eq(fold_name)].copy()
        cfg = config_from_recipe(recipe, feature_frames[fold_name])
        config_rows.append({'candidate': candidate, 'fold': fold_name, 'enabled': bool(recipe.get('enabled', False)), 'note': recipe.get('note', ''), **cfg.__dict__})
        revised = apply_trend_participation_target_gate(fold_targets, feature_frames[fold_name], cfg)
        revised['candidate'] = candidate
        revised['target_recipe'] = candidate
        fold_frames.append(revised)
    targets = pd.concat(fold_frames, ignore_index=True).sort_values(['fold', 'datetime']).reset_index(drop=True)
    candidate_targets[candidate] = targets
    by_fold = targets.groupby('fold').agg(
        rows=('datetime', 'count'),
        trend_target_rows=('trend_participation_target_state', 'sum'),
        rebound_rows=('is_rebound_cluster', 'sum'),
        downtrend_rows=('is_downtrend_state', 'sum'),
        mean_teacher_position=('teacher_position', 'mean'),
        mean_rebound_target_position=('rebound_target_position', 'mean'),
        mean_risk_target_position=('risk_target_position', 'mean'),
        max_objective_weight=('objective_weight', 'max'),
    ).reset_index()
    by_fold.insert(0, 'candidate', candidate)
    variant_rows.append(by_fold)

candidate_configs = pd.DataFrame(config_rows)
target_variant_summary = pd.concat(variant_rows, ignore_index=True)
candidate_configs.to_csv(CARD / 'candidate_configs.csv', index=False)
target_variant_summary.to_csv(CARD / 'target_variant_summary.csv', index=False)
target_variant_summary


,candidate,fold,rows,trend_target_rows,rebound_rows,downtrend_rows,mean_teacher_position,mean_rebound_target_position,mean_risk_target_position,max_objective_weight
0,base_foldlocal_contract,wf_2022_train,505,0,66,234,0.379034,0.379034,0.348817,366.115581
1,base_foldlocal_contract,wf_2023_2024_train,756,0,66,478,0.297715,0.297715,0.277530,252.245919
2,trend_q70_target45_guard,wf_2022_train,505,0,66,234,0.379034,0.379034,0.348817,366.115581
3,trend_q70_target45_guard,wf_2023_2024_train,756,15,81,478,0.297715,0.301485,0.277530,252.245919
4,trend_q75_target50_guard,wf_2022_train,505,0,66,234,0.379034,0.379034,0.348817,366.115581
5,trend_q75_target50_guard,wf_2023_2024_train,756,14,80,478,0.297715,0.302157,0.277530,252.245919
6,trend_q80_target55_guard,wf_2022_train,505,0,66,234,0.379034,0.379034,0.348817,366.115581
7,trend_q80_target55_guard,wf_2023_2024_train,756,8,74,478,0.297715,0.300830,0.277530,252.245919


In [5]:
audit_rows = []
lift_rows = []
lift_summary_rows = []
profile_rows = []
bc_score_rows = []
selection_rows = []
for candidate, targets in candidate_targets.items():
    scope_audit = audit_state_objective_target_scope(targets, folds=folds)
    scope_audit.insert(0, 'candidate', candidate)
    scope_summary = summarize_state_objective_target_scope_audit(scope_audit.drop(columns=['candidate']))
    lift_audit = audit_state_objective_target_lift(
        targets,
        group_cols=('fold',),
        baseline_risk_position_col='risk_target_position_original',
    )
    lift_audit.insert(0, 'candidate', candidate)
    lift_summary = summarize_state_objective_target_lift_audit(lift_audit.drop(columns=['candidate']))
    profile_dist = target_action_distribution(targets)
    profile_dist.insert(0, 'candidate', candidate)
    bc_scores = score_state_objective_bc_readiness(
        profile_dist.rename(columns={'profile': 'profile'}).drop(columns=['candidate']),
        target_scope=TARGET_SCOPE,
    )
    bc_scores.insert(0, 'candidate', candidate)
    bc_summary = summarize_state_objective_bc_readiness(
        bc_scores[bc_scores['profile'].eq('all_folds_trend_contract')].drop(columns=['candidate'])
    )
    trend_rows = int(targets['trend_participation_target_state'].fillna(False).astype(bool).sum()) if 'trend_participation_target_state' in targets.columns else 0
    trend_ratio = trend_rows / max(len(targets), 1)
    trend_fold_counts = targets.groupby('fold')['trend_participation_target_state'].sum().to_dict() if 'trend_participation_target_state' in targets.columns else {}
    trend_gate = bool((candidate == 'base_foldlocal_contract') or (10 <= trend_rows <= max(10, int(0.18 * len(targets)))))
    training_ready = bool(
        scope_summary['training_allowed']
        and lift_summary['training_allowed']
        and bc_summary['selected_bc_readiness_gate']
        and trend_gate
        and candidate != 'base_foldlocal_contract'
    )
    readiness_loss = (
        (0 if scope_summary['training_allowed'] else 1000)
        + (0 if lift_summary['training_allowed'] else 500)
        + (0 if bc_summary['selected_bc_readiness_gate'] else 100)
        + (0 if trend_gate else 50)
        + float(bc_summary.get('selected_bc_readiness_loss', 0.0))
        + abs(trend_ratio - 0.05) * 10.0
    )
    selection_rows.append({
        'candidate': candidate,
        'training_ready': training_ready,
        'readiness_loss': float(readiness_loss),
        'scope_decision': scope_summary['decision'],
        'lift_decision': lift_summary['decision'],
        'bc_decision': bc_summary['decision'],
        'scope_gate': bool(scope_summary['training_allowed']),
        'lift_gate': bool(lift_summary['training_allowed']),
        'bc_readiness_gate': bool(bc_summary['selected_bc_readiness_gate']),
        'trend_gate': trend_gate,
        'trend_target_rows': trend_rows,
        'trend_target_ratio': float(trend_ratio),
        'trend_fold_counts_json': json.dumps(json_safe(trend_fold_counts), sort_keys=True),
        'selected_unique_actions': int(bc_summary['selected_unique_actions']),
        'selected_dominant_action_share': float(bc_summary['selected_dominant_action_share']),
        'current_positive_rebound_lift_rows': int(lift_summary['current_positive_rebound_lift_rows']),
        'candidate_release_positive_lift_rows': int(lift_summary['candidate_release_positive_lift_rows']),
    })
    audit_rows.append(scope_audit)
    lift_rows.append(lift_audit)
    lift_summary_rows.append({'candidate': candidate, **lift_summary})
    profile_rows.append(profile_dist)
    bc_score_rows.append(bc_scores)

target_scope_audit = pd.concat(audit_rows, ignore_index=True)
target_lift_audit = pd.concat(lift_rows, ignore_index=True)
target_lift_summary = pd.DataFrame(lift_summary_rows)
profile_label_comparison = pd.concat(profile_rows, ignore_index=True)
bc_readiness_scores = pd.concat(bc_score_rows, ignore_index=True)
readiness_scores = pd.DataFrame(selection_rows).sort_values(['training_ready', 'readiness_loss', 'candidate'], ascending=[False, True, True]).reset_index(drop=True)

target_scope_audit.to_csv(CARD / 'target_scope_audit.csv', index=False)
target_lift_audit.to_csv(CARD / 'target_lift_audit.csv', index=False)
target_lift_summary.to_csv(CARD / 'target_lift_summary.csv', index=False)
profile_label_comparison.to_csv(CARD / 'profile_label_comparison.csv', index=False)
bc_readiness_scores.to_csv(CARD / 'bc_readiness_scores.csv', index=False)
readiness_scores.to_csv(CARD / 'readiness_scores.csv', index=False)
readiness_scores


,candidate,training_ready,readiness_loss,scope_decision,lift_decision,bc_decision,scope_gate,lift_gate,bc_readiness_gate,trend_gate,trend_target_rows,trend_target_ratio,trend_fold_counts_json,selected_unique_actions,selected_dominant_action_share,current_positive_rebound_lift_rows,candidate_release_positive_lift_rows
0,trend_q70_target45_guard,True,0.381047,state_objective_targets_training_ready,state_objective_target_lift_training_ready,state_objective_bc_ready_for_training,True,True,True,True,15,0.011895,"{""wf_2022_train"": 0, ""wf_2023_2024_train"": 15}",18,0.220460,95,129
1,trend_q75_target50_guard,True,0.388977,state_objective_targets_training_ready,state_objective_target_lift_training_ready,state_objective_bc_ready_for_training,True,True,True,True,14,0.011102,"{""wf_2022_train"": 0, ""wf_2023_2024_train"": 14}",18,0.220460,94,128
2,base_foldlocal_contract,False,0.500000,state_objective_targets_training_ready,state_objective_target_lift_training_ready,state_objective_bc_ready_for_training,True,True,True,True,0,0.000000,"{""wf_2022_train"": 0, ""wf_2023_2024_train"": 0}",18,0.221253,80,114
3,trend_q80_target55_guard,False,50.436558,state_objective_targets_training_ready,state_objective_target_lift_training_ready,state_objective_bc_ready_for_training,True,True,True,False,8,0.006344,"{""wf_2022_train"": 0, ""wf_2023_2024_train"": 8}",18,0.220460,88,122


In [6]:
selected = readiness_scores.iloc[0].to_dict() if not readiness_scores.empty else {}
selected_candidate = str(selected.get('candidate', ''))
selected_targets = candidate_targets[selected_candidate].copy() if selected_candidate else pd.DataFrame()
if not selected_targets.empty:
    selected_targets.to_csv(CARD / 'selected_trend_foldlocal_targets.csv', index=False)

if selected.get('training_ready', False):
    decision = 'trend_participation_foldlocal_targets_ready_for_bc_screen'
    next_mechanism = 'run_fold_local_bc_warm_start_then_walkforward_replay_with_drawdown_drift_stop'
    blocker = 'Training target readiness only; no actor has been trained or promoted, and fixed-split/wf replay is still required.'
    training_allowed = True
else:
    decision = 'trend_participation_foldlocal_targets_not_ready_no_training'
    next_mechanism = 'revise_trend_state_or_target_semantics_before_bc'
    blocker = 'No trend target variant jointly passed fold-local scope, target-lift, BC label-distribution, and trend-row sanity gates.'
    training_allowed = False

selection = {
    'experiment': EXPERIMENT,
    'created_at': datetime.now(timezone.utc).isoformat(),
    'decision': decision,
    'role': 'diagnose',
    'canonical_baseline': 'v19reg_v22_budget_usage_bull_guard_v1',
    'direct_comparison_baseline': [
        'trend_participation_guard_screen_v1',
        'cap10_sequence_risk_target_cap_bc_screen_v1:baseline_no_sequence_cap_target_contract',
    ],
    'profile': PROFILE_NAME,
    'changed_layer': ['model_selection', 'training_target'],
    'selected_candidate': selected_candidate,
    'training_allowed_from_this_card': bool(training_allowed),
    'promotion_allowed': False,
    'promotion_blocker': blocker,
    'next_mechanism': next_mechanism,
    'source_trend_screen_decision': trend_screen.get('decision'),
    'source_trend_screen_selected': trend_screen.get('selected_candidate'),
    'selected_readiness': selected,
    'external_sources': EXTERNAL_SOURCES,
    'sidecars': [
        'candidate_configs.csv',
        'target_variant_summary.csv',
        'target_scope_audit.csv',
        'target_lift_audit.csv',
        'target_lift_summary.csv',
        'profile_label_comparison.csv',
        'bc_readiness_scores.csv',
        'readiness_scores.csv',
        'selected_trend_foldlocal_targets.csv',
        'selection.json',
        'summary.json',
    ],
}
summary = {
    'experiment': EXPERIMENT,
    'decision': decision,
    'selected_candidate': selected_candidate,
    'training_allowed_from_this_card': bool(training_allowed),
    'promotion_allowed': False,
    'selected_readiness': selected,
    'next_mechanism': next_mechanism,
    'conclusion': [
        'Trend participation is now represented as fold-local target labels rather than a runtime overlay.',
        'The card only authorizes a next BC/checkpoint screen if scope, lift, BC-readiness, and trend-row sanity gates pass.',
        'Runtime promotion remains blocked because no actor has been trained or replayed from these targets.',
    ],
}
write_json(CARD / 'selection.json', selection)
write_json(CARD / 'summary.json', summary)
print(json.dumps(selection, indent=2))


{
  "experiment": "trend_participation_foldlocal_target_v1",
  "created_at": "2026-05-26T05:44:48.982317+00:00",
  "decision": "trend_participation_foldlocal_targets_ready_for_bc_screen",
  "role": "diagnose",
  "canonical_baseline": "v19reg_v22_budget_usage_bull_guard_v1",
  "direct_comparison_baseline": [
    "trend_participation_guard_screen_v1",
    "cap10_sequence_risk_target_cap_bc_screen_v1:baseline_no_sequence_cap_target_contract"
  ],
  "profile": "sparse_release_bc_ppo_screen_v1",
  "changed_layer": [
    "model_selection",
    "training_target"
  ],
  "selected_candidate": "trend_q70_target45_guard",
  "training_allowed_from_this_card": true,
  "promotion_allowed": false,
  "promotion_blocker": "Training target readiness only; no actor has been trained or promoted, and fixed-split/wf replay is still required.",
  "next_mechanism": "run_fold_local_bc_warm_start_then_walkforward_replay_with_drawdown_drift_stop",
  "source_trend_screen_decision": "trend_participation_guard_no_o

In [7]:
selection = json.loads((CARD / 'selection.json').read_text(encoding='utf-8'))
readiness_scores = pd.read_csv(CARD / 'readiness_scores.csv')
summary_md = f"""
## Runtime Conclusion

Decision: `{selection['decision']}`. Promotion is blocked.

Selected target candidate: `{selection['selected_candidate']}`. Training allowed from this target card: `{selection['training_allowed_from_this_card']}`.

The source runtime overlay result was `{selection['source_trend_screen_decision']}` with selected row `{selection['source_trend_screen_selected']}`. This notebook therefore keeps trend participation as a fold-local training/checkpoint objective, not an execution overlay.

Top readiness row:

```json
{json.dumps(selection['selected_readiness'], indent=2)}
```

Next action: `{selection['next_mechanism']}`.
""".strip()
display(Markdown(summary_md))
readiness_scores


## Runtime Conclusion

Decision: `trend_participation_foldlocal_targets_ready_for_bc_screen`. Promotion is blocked.

Selected target candidate: `trend_q70_target45_guard`. Training allowed from this target card: `True`.

The source runtime overlay result was `trend_participation_guard_no_objective_pass_no_promotion` with selected row `base_seq_cap10_two_of_three`. This notebook therefore keeps trend participation as a fold-local training/checkpoint objective, not an execution overlay.

Top readiness row:

```json
{
  "bc_decision": "state_objective_bc_ready_for_training",
  "bc_readiness_gate": true,
  "candidate": "trend_q70_target45_guard",
  "candidate_release_positive_lift_rows": 129,
  "current_positive_rebound_lift_rows": 95,
  "lift_decision": "state_objective_target_lift_training_ready",
  "lift_gate": true,
  "readiness_loss": 0.38104678826328314,
  "scope_decision": "state_objective_targets_training_ready",
  "scope_gate": true,
  "selected_dominant_action_share": 0.2204599524187153,
  "selected_unique_actions": 18,
  "training_ready": true,
  "trend_fold_counts_json": "{\"wf_2022_train\": 0, \"wf_2023_2024_train\": 15}",
  "trend_gate": true,
  "trend_target_ratio": 0.011895321173671689,
  "trend_target_rows": 15
}
```

Next action: `run_fold_local_bc_warm_start_then_walkforward_replay_with_drawdown_drift_stop`.

,candidate,training_ready,readiness_loss,scope_decision,lift_decision,bc_decision,scope_gate,lift_gate,bc_readiness_gate,trend_gate,trend_target_rows,trend_target_ratio,trend_fold_counts_json,selected_unique_actions,selected_dominant_action_share,current_positive_rebound_lift_rows,candidate_release_positive_lift_rows
0,trend_q70_target45_guard,True,0.381047,state_objective_targets_training_ready,state_objective_target_lift_training_ready,state_objective_bc_ready_for_training,True,True,True,True,15,0.011895,"{""wf_2022_train"": 0, ""wf_2023_2024_train"": 15}",18,0.220460,95,129
1,trend_q75_target50_guard,True,0.388977,state_objective_targets_training_ready,state_objective_target_lift_training_ready,state_objective_bc_ready_for_training,True,True,True,True,14,0.011102,"{""wf_2022_train"": 0, ""wf_2023_2024_train"": 14}",18,0.220460,94,128
2,base_foldlocal_contract,False,0.500000,state_objective_targets_training_ready,state_objective_target_lift_training_ready,state_objective_bc_ready_for_training,True,True,True,True,0,0.000000,"{""wf_2022_train"": 0, ""wf_2023_2024_train"": 0}",18,0.221253,80,114
3,trend_q80_target55_guard,False,50.436558,state_objective_targets_training_ready,state_objective_target_lift_training_ready,state_objective_bc_ready_for_training,True,True,True,False,8,0.006344,"{""wf_2022_train"": 0, ""wf_2023_2024_train"": 8}",18,0.220460,88,122
